# EDA — Camada Gold (modelo estrela, dbt-trino)

A Gold é construída pelo `dbt` (`dbt/models/`) e materializada como tabelas Iceberg em `iceberg.gold.*` — dimensões e fatos, com testes de qualidade rodando a cada `dbt build` (32/32 PASS na última validação). Este notebook consulta o resultado final, o mesmo que serviria um BI.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import trino
from dotenv import load_dotenv
import os

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(dotenv_path=project_root / ".env")

pd.set_option("display.max_columns", 40)

conn = trino.dbapi.connect(
    host=os.environ.get("TRINO_HOST", "trino"),
    port=int(os.environ.get("TRINO_PORT", 8080)),
    user=os.environ.get("TRINO_USER", "notebook"),
    http_scheme=os.environ.get("TRINO_HTTP_SCHEME", "http"),
    catalog=os.environ.get("TRINO_CATALOG", "iceberg"),
    schema="gold",
)


def query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [c[0] for c in cur.description]
    return pd.DataFrame(rows, columns=cols)


query("SHOW TABLES FROM iceberg.gold")


## 1. Volume por tabela do modelo estrela


In [ ]:
tabelas = ["fato_empenho", "fato_contrato", "dim_credor", "dim_orgao", "dim_modalidade", "dim_tempo"]
contagens = query(
    " UNION ALL ".join(
        f"SELECT '{t}' AS tabela, COUNT(*) AS registros FROM iceberg.gold.{t}" for t in tabelas
    )
)
contagens


## 2. Cobertura de join — quantas linhas do fato ficam sem dimensão?

`fato_contrato` é o caso conhecido com FKs sem match (documentado como ~0,1% sem `sk_orgao`) — os testes dbt `relationships` ignoram nulo, então essa perda é silenciosa se ninguém checar manualmente.


In [ ]:
query(
    """
    SELECT
        COUNT(*) AS total_contratos,
        COUNT(*) FILTER (WHERE sk_orgao IS NULL) AS sem_orgao,
        COUNT(*) FILTER (WHERE sk_credor IS NULL) AS sem_credor,
        ROUND(100.0 * COUNT(*) FILTER (WHERE sk_orgao IS NULL) / COUNT(*), 2) AS pct_sem_orgao
    FROM iceberg.gold.fato_contrato
    """
)


## 3. Negócio — top 10 órgãos por valor empenhado


In [ ]:
query(
    """
    SELECT
        o.nome_orgao,
        COUNT(*) AS qtd_empenhos,
        ROUND(SUM(f.valor), 2) AS valor_total
    FROM iceberg.gold.fato_empenho f
    JOIN iceberg.gold.dim_orgao o ON f.sk_orgao = o.sk_orgao
    GROUP BY o.nome_orgao
    ORDER BY valor_total DESC
    LIMIT 10
    """
)


## 4. `dim_credor` em SCD2 — quantos credores têm histórico?

SCD2 só faz sentido se houver credores com mais de uma versão registrada. Compara o total de linhas (versões) com o total de credores distintos.


In [ ]:
query(
    """
    SELECT
        COUNT(*) AS total_versoes,
        COUNT(DISTINCT cpf_cnpj) AS credores_distintos,
        COUNT(*) FILTER (WHERE versao_atual = true) AS versoes_atuais
    FROM iceberg.gold.dim_credor
    """
)


## Achados rápidos

- `pct_sem_orgao`/`pct_sem_credor` deve ficar próximo do ~0,1% já documentado — se subir muito, é sinal de regressão no join da Silver ou na origem.
- O ranking de órgãos por valor empenhado é o tipo de consulta que um BI faria direto em cima desta mesma tabela — a Gold já está pronta para servir.
- `versoes_atuais` deve ser igual a `credores_distintos` (uma versão atual por credor) e `total_versoes` >= `credores_distintos` (SCD2 acumulando histórico).
